# Setup

In [1]:
# install personal package for use below, type the below into the docker terminal
# pip install /tf/pyGroupSequentialDesigns/

In [2]:
# higher resolution graphs
%config InlineBackend.figure_format='retina'

## Published package imports

In [3]:
# imports for study design step (Step 1)
import numpy as np
import pandas as pd
from scipy import stats

# imports for GP regression (Step 3)
import gpflow

# imports for Bayes opt (Step 4-6)
import trieste
from trieste.space import Box
from trieste.models.gpflow.models import GaussianProcessRegression
import tensorflow as tf

/usr/local/lib/python3.11/dist-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
/usr/local/lib/python3.11/dist-packages/gpflow/versions.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## Personal package imports

In [4]:
from py_group_sequential_designs import boundaries as bd
from py_group_sequential_designs import feasibility_penalty as fp
from py_group_sequential_designs import format_boundaries_after_ask as fmt_bd
from py_group_sequential_designs import function_to_minimize as fn_min
from py_group_sequential_designs import generate_gpr_input as gen_input
from py_group_sequential_designs import simulate as sim
from py_group_sequential_designs import sample_size as ss

# Bayesian optimization workflow default values

In [5]:
# some set defaults
num_analyses = 3
target_alpha = 0.05
target_power = 0.9
important_diff_delta = 1
assumed_variance = 3

# to obtain mu (sample size at one stage)
# assume 1:1 randomization
group_ratio = 1

# Simulate initial points for GPR

## Create a helper function

In [6]:
# create a function that generates the points x and y that will be
# included in the design matrix X and Y
def generate_x_y(
        upper_bounds,
        lower_bounds,
        n_analyses,
        alt_hypothesis,
        variance,
        alpha_prime,
        ratio,
        target_power,
        target_alpha):

    # 1. Find the number of patients that achieves 90% power (beta 0.1)
    # here we get beta_prime and alpha prime
    n_power09, beta_prime = ss.find_sample_size(
        n_analyses=n_analyses,
        upper_bounds=upper_bounds,
        lower_bounds=lower_bounds,
        alt_hypothesis=important_diff_delta,
        variance=variance
    )
    
    # 2. Generate the GPR input values
    # note that the input includes the sample size at power 0.9
    x = gen_input.generate_gpr_input(
        n_analyses = n_analyses,
        upper_bounds=upper_bounds,
        lower_bounds=lower_bounds,
        n_patients=n_power09)
    
    # 3. Generate maximum expected sample size and feasibility penalty
    max_ess_new = ss.max_ess(
        n_analyses=n_analyses,
        upper_bounds=upper_bounds,
        lower_bounds=lower_bounds,
        n_patients=n_power09)
    
    penalty = fp.feasibility_penalty(
        ratio=ratio,
        variance=variance,
        power=target_power,
        alpha=target_alpha,
        delta=important_diff_delta,
        beta_prime=beta_prime,
        alpha_prime=alpha_prime
    )
    
    
    # 4. Calculate the function value (GPR output)
    y = fn_min.function_to_minimize(max_ess_val=max_ess_new, penalty=penalty)

    return (np.array([x]), np.array([[y]]))

## Point 1

In [7]:
# simulate the trial design 
poc_simulation = bd.calculate_pocock_boundaries(
    n_analyses=num_analyses,
    alpha=target_alpha,
    n_patients=20
)

x1, y1 = generate_x_y(
    upper_bounds = poc_simulation[0],
    lower_bounds = poc_simulation[1],
    n_analyses = num_analyses,
    alt_hypothesis = important_diff_delta,
    variance = assumed_variance,
    alpha_prime = poc_simulation[3],
    ratio = group_ratio,
    target_power = target_power,
    target_alpha = target_alpha
)

## Point 2

In [8]:
# simulate the trial design 
of_simulation = bd.calculate_of_boundaries(
    n_analyses=num_analyses,
    alpha=target_alpha,
    n_patients=20
)

x2, y2 = generate_x_y(
    upper_bounds = of_simulation[0],
    lower_bounds = of_simulation[1],
    n_analyses = num_analyses,
    alt_hypothesis = important_diff_delta,
    variance = assumed_variance,
    alpha_prime = of_simulation[3],
    ratio = group_ratio,
    target_power = target_power,
    target_alpha = target_alpha
)

## Point 3

In [9]:
# simulate the trial design 
simulation = bd.calculate_triangular_boundaries(
    n_analyses=num_analyses,
    alpha=target_alpha,
    delta=important_diff_delta,
    n_patients=20
)

x3, y3 = generate_x_y(
    upper_bounds = simulation[0],
    lower_bounds = simulation[1],
    n_analyses = num_analyses,
    alt_hypothesis = important_diff_delta,
    variance = assumed_variance,
    alpha_prime = simulation[3],
    ratio = group_ratio,
    target_power = target_power,
    target_alpha = target_alpha
)

# Bayesian optimization loop

## Enter the initial points

In [10]:
design_matrix = np.concatenate((x1, x2, x3))
design_matrix

array([[ 1.99218586e+00,  1.99218586e+00,  1.99218586e+00,
        -1.99218586e+00, -1.99218586e+00,  1.99629163e+01],
       [ 2.96113074e+00,  2.09383563e+00,  1.70960963e+00,
        -2.96113074e+00, -2.09383563e+00,  1.75540968e+01],
       [ 2.11957748e+00,  1.87345951e+00,  1.83560794e+00,
         6.28553399e-16,  1.12407571e+00,  2.13211856e+01]])

In [11]:
output_vals = np.concatenate((y1, y2, y3))
output_vals

array([[57.41979824],
       [51.95056086],
       [45.46802067]])

In [12]:
def build_model(X, Y):
    
    kernel = gpflow.kernels.Matern52(variance=1)

    likelihood = gpflow.likelihoods.Gaussian()
        
    gpr = gpflow.models.GPR(
        data = (X, Y),
        kernel = kernel,
        likelihood = likelihood
    )

    gpflow.utilities.print_summary(gpr, fmt="notebook")
    
    return GaussianProcessRegression(gpr)

In [13]:
bayes_opt_model = build_model(X = design_matrix, Y = output_vals)

name,class,transform,prior,trainable,shape,dtype,value
GPR.kernel.variance,Parameter,Softplus,,True,(),float64,1
GPR.kernel.lengthscales,Parameter,Softplus,,True,(),float64,1
GPR.likelihood.variance,Parameter,Softplus + Shift,,True,(),float64,1
